# Étape 10 — Phase E : MLP stacker 

## 1. Setup

In [1]:
import sys, pickle, gc
from pathlib import Path

ROOT = Path.cwd()   # racine du depot (les notebooks sont a la racine)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = ROOT / 'data'
EXP_DIR  = ROOT / 'experiments'
ART_DIR  = ROOT / 'artifacts'
SUB_DIR  = ROOT / 'submissions'; SUB_DIR.mkdir(exist_ok=True)

with open(ART_DIR / 'pipeline_phaseE.pkl', 'rb') as f:
    pipe = pickle.load(f)
print('phase     :', pipe['phase'])
print('CV blend3 :', pipe['val_score_5foldCV_blend3']['score'])
print('CV mlp    :', pipe['val_score_5foldCV_mlp']['score'])
print('blend_3   :', pipe['blend_3'])
print('mlp arch  :', pipe['mlp_stacker']['arch'])
print('n_ckpts   :', len(pipe['mlp_stacker']['ckpt_files']))

phase     : E
CV blend3 : 0.000888046742288194
CV mlp    : 0.000888046742288194
blend_3   : {'am_mlp': 1.0, 'ast_lgbm': 0.0, 'aw_weighted': 0.0}
mlp arch  : {'hidden': 64, 'drop': 0.1, 'in_dim': 14}
n_ckpts   : 25


## 2. Recharge les prédictions test (Phase D) et df_test

In [2]:
arc_test = np.load(EXP_DIR / 'phaseD_preds_test.npz')
FILENAMES_TEST = list(arc_test['filenames'])
names = pipe['model_names']
P_test = np.stack([arc_test[n] for n in names])
print('P_test shape:', P_test.shape)

# Reconstitue df_test_full + gender via identity-join
df_test_full = pd.read_csv(DATA_DIR / 'test_clean.csv')
df_train     = pd.read_csv(DATA_DIR / 'train_clean.csv')
gender_by_id = (df_train.groupby('identity')['gender']
                .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan))
fallback_gender = float(df_train['gender'].mean())
df_test_full['gender'] = (df_test_full['identity'].map(gender_by_id).astype(float)
                          .fillna(fallback_gender))

df_test = df_test_full[~df_test_full.needs_imputation].reset_index(drop=True).copy()
df_imp  = df_test_full[ df_test_full.needs_imputation].reset_index(drop=True).copy()
assert df_test['filename'].tolist() == FILENAMES_TEST, 'filename order mismatch'
print(f'df_test : {len(df_test)} | df_imp : {len(df_imp)}')
print(f'gender mapped via identity : {(df_test_full["gender"] != fallback_gender).sum()}/{len(df_test_full)}')

P_test shape: (7, 29862)


df_test : 29862 | df_imp : 118
gender mapped via identity : 28036/29980


## 3. Build features + standardisation

In [3]:
feature_columns = pipe['mlp_stacker']['feature_columns']
scaler = np.load(pipe['mlp_stacker']['feat_scaler_file'])
mu, sd = scaler['mu'], scaler['sd']

def build_features(P_arr, gender_arr, names_):
    feats = {f'pred_{names_[i]}': P_arr[i] for i in range(len(names_))}
    feats['gender']   = gender_arr.astype(np.float32)
    feats['p_mean']   = P_arr.mean(0)
    feats['p_std']    = P_arr.std(0)
    feats['p_min']    = P_arr.min(0)
    feats['p_max']    = P_arr.max(0)
    feats['p_range']  = feats['p_max'] - feats['p_min']
    feats['p_median'] = np.median(P_arr, 0)
    return pd.DataFrame(feats)[feature_columns].values.astype(np.float32)

gender_test = df_test['gender'].values.astype(np.float32)
X_test = build_features(P_test, gender_test, names)
X_test_n = ((X_test - mu) / sd).astype(np.float32)
print('X_test_n :', X_test_n.shape)

X_test_n : (29862, 14)


## 4. Charge MLP et prédiction en moyenne

In [4]:
class MLPStacker(nn.Module):
    def __init__(self, in_dim, hidden=64, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(-1)

arch = pipe['mlp_stacker']['arch']
ckpt_files = pipe['mlp_stacker']['ckpt_files']

X_test_t = torch.from_numpy(X_test_n).to(device)
preds = []
for cf in ckpt_files:
    m = MLPStacker(in_dim=arch['in_dim'], hidden=arch['hidden'], drop=arch['drop']).to(device)
    state = torch.load(cf, map_location=device, weights_only=False)
    m.load_state_dict(state['state'])
    m.eval()
    with torch.no_grad():
        p = m(X_test_t).cpu().numpy()
    preds.append(p)
    del m
torch.cuda.empty_cache(); gc.collect()
preds = np.stack(preds)
print(f'preds shape : {preds.shape}  ({len(ckpt_files)} ckpts × {len(X_test_n)} samples)')
print(f'  pred mean per ckpt: {preds.mean(axis=1).round(4)}')

mlp_pred = preds.mean(axis=0)
mlp_pred = np.clip(mlp_pred, 0.0, 1.0).astype(np.float32)
print(f'\nmlp_pred (avg of {len(ckpt_files)}) : mean={mlp_pred.mean():.4f} | std={mlp_pred.std():.4f} | min={mlp_pred.min():.4f} | max={mlp_pred.max():.4f}')

preds shape : (25, 29862)  (25 ckpts × 29862 samples)
  pred mean per ckpt: [0.1696 0.1585 0.1624 0.1738 0.1659 0.1644 0.1742 0.1605 0.1644 0.1668
 0.1756 0.1705 0.1748 0.171  0.1661 0.1614 0.1698 0.1649 0.1647 0.1687
 0.1696 0.1582 0.1722 0.1594 0.1615]

mlp_pred (avg of 25) : mean=0.1667 | std=0.1018 | min=0.0109 | max=0.7603


## 5. Blend final

In [5]:
import lightgbm as lgb

# --- LGBM stacking (Phase C) ---
X_test_df = pd.DataFrame(X_test, columns=feature_columns)
stk_pred = np.zeros(len(df_test), dtype=np.float64)
for path in pipe['stacking']['boosters_files']:
    booster = lgb.Booster(model_file=path)
    stk_pred += booster.predict(X_test_df, num_iteration=booster.best_iteration)
stk_pred /= len(pipe['stacking']['boosters_files'])
stk_pred = np.clip(stk_pred, 0.0, 1.0).astype(np.float32)

# --- weighted ensemble ---
weights = np.array([next(m['weight'] for m in pipe['models'] if m['name'] == n) for n in names], dtype=np.float64)
weights = weights / weights.sum()
w_pred = (weights[:, None] * P_test).sum(axis=0)
w_pred = np.clip(w_pred, 0.0, 1.0).astype(np.float32)

# --- blend final selon pipe['blend_3'] ---
am  = pipe['blend_3']['am_mlp']
ast = pipe['blend_3']['ast_lgbm']
aw  = pipe['blend_3']['aw_weighted']
blend = am*mlp_pred + ast*stk_pred + aw*w_pred
blend = np.clip(blend, 0.0, 1.0).astype(np.float32)
print(f'3-blend (am={am:.3f}, ast={ast:.3f}, aw={aw:.3f})')
print(f'  mean={blend.mean():.4f} | std={blend.std():.4f} | min={blend.min():.4f} | max={blend.max():.4f}')

# --- alt : MLP seul (sub de secours) ---
mlp_only = mlp_pred.copy()

3-blend (am=1.000, ast=0.000, aw=0.000)
  mean=0.1667 | std=0.1018 | min=0.0109 | max=0.7603


## 6. Imputation des 118 lignes sans image + save submission

In [ ]:
_w = (1.0 / 30.0 + df_train['FaceOcclusion'].values)
IMPUT_VALUE = float(np.sum(_w * df_train['FaceOcclusion'].values) / np.sum(_w))
print(f'IMPUT_VALUE = {IMPUT_VALUE:.5f}')

df_students = pd.read_csv(DATA_DIR / 'test_students.csv')

def make_sub(pred_array, name):
    preds_full = dict(zip(df_test['filename'].tolist(), pred_array.astype(float).tolist()))
    for fn in df_imp['filename'].tolist():
        preds_full[fn] = IMPUT_VALUE
    assert set(df_students['filename']) == set(preds_full.keys())
    sub = pd.DataFrame({
        'filename': df_students['filename'],
        'FaceOcclusion': df_students['filename'].map(preds_full),
        'gender': 'x',
    })
    assert sub['FaceOcclusion'].notna().all()
    assert sub['FaceOcclusion'].between(0, 1).all()
    out_path = SUB_DIR / name
    sub.to_csv(out_path, index=False)
    print(f'  saved {out_path}  | n={len(sub)} | mean={sub.FaceOcclusion.mean():.4f}')
    return sub, out_path

# Soumission principale
sub_mlp, path_mlp = make_sub(mlp_only, 'submission_phaseE_mlp.csv')
# Backup
sub_blend3, path_blend3 = make_sub(blend, 'submission_phaseE_blend3.csv')

# Diff entre les 2 (devrait être 0 si am=1)
diff = (sub_mlp.FaceOcclusion - sub_blend3.FaceOcclusion).abs()
print(f'\nMLP vs blend3 : max diff = {diff.max():.6f}')

IMPUT_VALUE = 0.14699
  saved <repo>/submissions/submission_phaseE_mlp.csv  | n=29980 | mean=0.1667


  saved <repo>/submissions/submission_phaseE_blend3.csv  | n=29980 | mean=0.1667

MLP vs blend3 : max diff = 0.000000
